# Reproduce the Panel Benchmark

This notebook reproduces the main multi-country panel benchmark for **Event-Informed Port Disruption Prediction with Network-Weighted Trade Exposure**.

It is designed as a clean public reproduction entry point. The underlying data and modeling logic lives in `src/` and `scripts/`, while this notebook documents the research workflow and executes the pipeline step by step.

## Research Design

The benchmark asks whether external event signals improve next-week abnormal container port activity prediction beyond operational baselines, and whether trade-network-weighted exposure adds value over unweighted event controls.

The model ladder is:

- **M1**: operational baseline.
- **M2**: M1 plus own-country GDELT event controls.
- **M3**: M2 plus external unweighted partner-event controls.
- **M4**: M3 plus total-import network-weighted exposure.
- **M5**: M3 plus machinery/electronics strict network exposure.
- **M6**: placebo variants using equal, shuffled, or random weights.

Evaluation uses temporal rolling-origin validation. PR-AUC is the primary metric because abnormal port activity is rare.

## Prerequisites

The public repository does not commit large raw/interim data files. To run this notebook end to end, the following cached GDELT files must exist under `data/interim/`:

- `gkg_partner_event_features_2021-01-01_2025-12-31.csv`
- `gkg_partner_me_strict_event_features_2021-01-01_2025-12-31.csv`

These files are generated from partition-filtered GDELT GKG BigQuery queries. PortWatch and WITS data are fetched by the dataset builder.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])

In [ ]:
required_cache_files = [
    PROJECT_ROOT / "data" / "interim" / "gkg_partner_event_features_2021-01-01_2025-12-31.csv",
    PROJECT_ROOT / "data" / "interim" / "gkg_partner_me_strict_event_features_2021-01-01_2025-12-31.csv",
]

missing = [path for path in required_cache_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing cached GDELT feature files. Generate or place these files first:\n"
        + "\n".join(str(path) for path in missing)
    )

for path in required_cache_files:
    size_mb = path.stat().st_size / 1_000_000
    print(f"Found {path.name}: {size_mb:.1f} MB")

## Step 1: Build the Benchmark Dataset

This step constructs the model-ready country-week panel. It fetches PortWatch weekly container activity, builds WITS import-dependency weights, loads cached GDELT event features, computes unweighted and network-weighted exposures, and saves the processed benchmark dataset.

In [ ]:
def run_script(script_name):
    command = [sys.executable, str(PROJECT_ROOT / "scripts" / script_name)]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)


run_script("build_panel_benchmark_dataset.py")

In [ ]:
dataset_path = PROJECT_ROOT / "data" / "processed" / "multicountry_container_event_network_benchmark.csv"
df = pd.read_csv(dataset_path, parse_dates=["week"])

summary = {
    "rows": len(df),
    "countries": df["ISO3"].nunique(),
    "positive_labels": int(df["abnormal_next_week_container"].sum()),
    "positive_rate": round(df["abnormal_next_week_container"].mean(), 4),
    "min_week": df["week"].min().date(),
    "max_week": df["week"].max().date(),
}
summary

## Step 2: Run the M1-M6 Benchmark Models

This step evaluates balanced Logistic Regression and Random Forest models using temporal rolling-origin splits. Thresholds are selected on validation folds only, then applied to test folds.

In [ ]:
run_script("run_panel_benchmark_models.py")

In [ ]:
summary_path = PROJECT_ROOT / "reports" / "tables" / "panel_benchmark_summary.csv"
model_summary = pd.read_csv(summary_path)

display(
    model_summary.sort_values(["model", "mean_pr_auc"], ascending=[True, False])[
        [
            "feature_group",
            "model",
            "mean_pr_auc",
            "std_pr_auc",
            "mean_roc_auc",
            "mean_f1",
            "mean_precision",
            "mean_recall",
            "total_tp",
            "total_fp",
            "total_fn",
        ]
    ].head(20)
)

## Step 3: Generate Figures

This step creates the main benchmark figures under `reports/figures/`, including model comparison, fold stability, exposure time series, and network overview figures.

In [ ]:
for script in [
    "make_panel_benchmark_figures.py",
    "make_supply_chain_network_overview.py",
]:
    run_script(script)

In [ ]:
figure_dir = PROJECT_ROOT / "reports" / "figures"
figures = sorted(path.name for path in figure_dir.glob("fig_*.png"))
figures

## Step 4: Final Reproducibility Check

The tests are intentionally lightweight and offline. They check schema, temporal split ordering, leakage guardrails, and deterministic placebo behavior.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests"],
    cwd=PROJECT_ROOT,
    check=True,
)

## Expected Outputs

Main generated artifacts:

- `data/processed/multicountry_container_event_network_benchmark.csv`
- `reports/panel_benchmark_dataset_summary.md`
- `reports/panel_benchmark_results.md`
- `reports/tables/panel_benchmark_summary.csv`
- `reports/tables/panel_benchmark_metrics_by_fold.csv`
- `reports/figures/fig_panel_model_comparison_pr_auc.png`
- `reports/figures/fig_panel_pr_auc_by_fold.png`
- `reports/figures/fig_all_country_supply_chain_network.png`

Interpret results cautiously: the network layer is an exposure-mapping mechanism, not a causal effect estimate.